
# Milestone 3: Pyridine Parameter Estimation

## Overview
Here, we put everything together to estimate the parameters of the Pyridine problem:
- Use the multiple shooting method
- Apply CNLLS solvers to real measurement data

## Setup

In [ ]:
from dynamics import create_integrator, pyridine_ode
from multiple_shooting import setup_multiple_shooting
from cnlls_solver import solve_cnlls_ipopt, solve_cnlls_gauss_newton
from utils import load_measurements

nx, np_p, N, dt = 3, 2, 20, 0.5
integrator = create_integrator(nx, np_p, pyridine_ode, dt)

# Set up multiple shooting
w, X_end, g, S_vars, P_var = setup_multiple_shooting(integrator, N, nx, np_p)

# Load measurements
measurements = load_measurements("../data/measurements.npy")

# Build F1 (measurement residuals)
F1 = ca.vertcat(*[X_end[i] - measurements[i] for i in range(N)])

In [ ]:
# Solve with IPOPT
w0 = np.random.rand(w.size1())
sol_ipopt = solve_cnlls_ipopt(w, F1, g, w0)
print("IPOPT estimated parameters:", sol_ipopt['x'][-np_p:].full().flatten())

In [ ]:
# Solve with Gauss-Newton
J_full = compute_full_jacobian(integrator, S_vars, P_var, N, nx, np_p)
sol_gn = solve_cnlls_gauss_newton(w, F1, g, w0, J_full)
print("Gauss-Newton estimated parameters:", sol_gn['x'][-np_p:].full().flatten())